# Decision tree lab: who is about to cancel?
**MBI806B · Business Data Analytics with AI and ML**

This is the finished notebook for the lab. Use it to check your own work if you get stuck, not instead of doing it. Your report needs screenshots of *your* notebook, with your name in the title.

**Before you run anything:** unzip the Kaggle download, rename the CSV to `telco.csv`, then click the folder icon on the left, then the upload button, and upload it. The file disappears when Colab disconnects, so upload it again if you come back later.

## Step 1: Load the data

In [ ]:
import pandas as pd

df = pd.read_csv("telco.csv")
df.head()

## Step 2: Get to know it

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe().round(1)

In [ ]:
df["Churn"].value_counts()

## Step 3: Clean it
**Why:** a model learns whatever it's given, mistakes included.

In [ ]:
df.isnull().sum()

Nothing missing? Look closer. TotalCharges should be a number, but pandas read it as text. Something in it isn't a number.

In [ ]:
print((df["TotalCharges"] == " ").sum())

In [ ]:
df[df["TotalCharges"] == " "][["tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

All 11 joined this month (tenure 0), so they haven't been billed yet. Turn the column into numbers, and give the blanks 0.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["TotalCharges"].dtype

In [ ]:
print(df["customerID"].duplicated().sum(), "repeated customer IDs")
print(df.drop(columns="customerID").duplicated().sum(), "rows that match another row apart from the ID")

A model only reads numbers. Turn the three columns we'll use into numbers.

In [ ]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df["Contract"] = df["Contract"].map({"Month-to-month": 0, "One year": 1, "Two year": 2})
df["Fiber"] = df["InternetService"].map({"Fiber optic": 1, "DSL": 0, "No": 0})
df[["Churn", "Contract", "InternetService", "Fiber"]].head()

## Step 4: Compare groups
The average of a column of 1s and 0s is the share of 1s. So the mean of Churn is the share of customers who left.

In [ ]:
df.groupby("Contract")["Churn"].mean().round(3)

In [ ]:
df.groupby("Fiber")["Churn"].mean().round(3)

## Step 5: Look before you model

In [ ]:
import matplotlib.pyplot as plt

rates = df.groupby("Contract")["Churn"].mean()
rates.plot(kind="bar", rot=0)
plt.xlabel("Contract: 0 = month-to-month, 1 = one year, 2 = two year")
plt.ylabel("Share who left")
plt.show()

In [ ]:
df.groupby("tenure")["Churn"].mean().plot()
plt.xlabel("Months as a customer")
plt.ylabel("Share who left")
plt.show()

## Step 6: Split, then train

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["tenure", "MonthlyCharges", "Contract", "Fiber"]]
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(len(X_train), "customers to learn from")
print(len(X_test), "customers kept back for the test")

The score to beat: guess "stays" for everyone.

In [ ]:
print("Always guessing 'stays':", round((y_test == 0).mean(), 3))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

print("Score:", round(tree.score(X_test, y_test), 3))

## Step 7: Read the tree

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(16, 7))
plot_tree(tree, feature_names=list(X.columns), class_names=["Stays", "Leaves"], filled=True, rounded=True)
plt.show()

In [ ]:
from sklearn.tree import export_text

print(export_text(tree, feature_names=list(X.columns)))

In [ ]:
predictions = tree.predict(X_test)
caught = ((predictions == 1) & (y_test == 1)).sum()

print("Customers in the test who really left:", y_test.sum())
print("Of those, the tree caught:", caught)

What happens if you let the tree grow as deep as it likes?

In [ ]:
big_tree = DecisionTreeClassifier(random_state=42)
big_tree.fit(X_train, y_train)

print("On the customers it learned from:", round(big_tree.score(X_train, y_train), 3))
print("On customers it has never seen:", round(big_tree.score(X_test, y_test), 3))

## Step 8: Predict someone new
Made-up customers only.

In [ ]:
new_customers = pd.DataFrame({
    "tenure":         [2, 40, 60],
    "MonthlyCharges": [95, 60, 25],
    "Contract":       [0, 1, 2],
    "Fiber":          [1, 0, 0],
})

print(tree.predict(new_customers))
print(tree.predict_proba(new_customers).round(2))

## Your turn
Change the three customers above to your own three made-up customers, run the cell again, and put the results in your report.